# Threat Record Manager

A CLI threat intelligence tool built in Python that simulates the record management a SOC analyst might perform on a daily basis.

What this notebook does:
- Views, updates, adds, deletes, and queries fields in a threat record
- Handles key renaming, value updates, and input validation
- Defensive design: cancel at any step with "q", no crashes on bad input
- Documents the build step by step: the initial version, each bug found once it ran, the fix applied, and the reasoning behind it

Built with: Python, os module, dictionaries, control flow, error handling


In [ ]:
threat = {
    "ip": "185.220.101.1",
    "country": "RU",
    "severity": "high",
    "alert_count": 5
}

while True:
    print("\n" + "="*41)
    print("="*9 + " THREAT RECORD MANAGER " + "="*9)
    print("="*41)
    print(" [1] View threat record\n",
          " [2] Update a field\n",
          " [3] Add a new field\n",
          " [4] Delete a field\n",
          " [5] Check if a field exists\n",
          " [Q] Quit\n",
          sep="")
    
    option = input("Enter option: ").lower()

    if option == "q":
        print("\nClosing...")
        break

    elif option == "1":
        print(f"\n{threat}")

    elif option == "2":

        try:
            key = str(input("Enter key: ")).lower()
            value = str(input("Enter value: ")).lower()
            
            for key, value in threat.items():
                new_key = str(input("Enter new key: "))
                new_value = str(input("Enter new value: "))

                threat[new_key] = new_value
                print("[Field updated]")

        except:
            print("Field not found")

### Revision 1: Bug Check - dictionary mutated during iteration
* When trying Option 2, I noticed that it asks for all the inputs but still throws the `except` error.
* I consulted Claude Chat to explain the issue and learned two things:
    1. You can't add or remove keys while actively looping in Python because it
will throw a `RuntimeError: dictionary changed size during iteration`.
    2. While it does update the old `key:value` pair, it will loop back and hit the said error, triggering the `except` block and printing `Field not found`.

**The fix:** the only reasonable fix is to use an `if-else` condition instead of a `try/except` block.

Updated script:


In [ ]:
threat = {
    "ip": "185.220.101.1",
    "country": "RU",
    "severity": "high",
    "alert_count": 5
}

while True:
    print("\n" + "="*41)
    print("="*9 + " THREAT RECORD MANAGER " + "="*9)
    print("="*41)
    print(" [1] View threat record\n",
          " [2] Update a field\n",
          " [3] Add a new field\n",
          " [4] Delete a field\n",
          " [5] Check if a field exists\n",
          " [Q] Quit\n",
          sep="")
    
    option = input("Enter option: ").lower()

    if option == "q":
        print("\nClosing...")
        break

    elif option == "1":
        print(f"\n{threat}")
        
    elif option == "2":
        old_key = input("Enter key: ").lower()

        if old_key in threat:
            new_key = input("Enter new key: ").lower()
            new_value = input("Enter new value: ").lower()

            if old_key != new_key:
                del threat[old_key]

            threat[new_key] = new_value
            print(f"\n[Field updated] Updated fields: {threat}")

        else:
            print(f"\nField {old_key} not found")

    elif option == "3":
        new_key = input("Enter new key: ").lower()

        if new_key not in threat:
            new_value = input("Enter new value: ").lower()

            threat[new_key] = new_value
            print(f"\n[Field added] Updated fields: {threat}")
            
        else:
            print("\nField already exists. Use option 2 to update.")

    elif option == "4":
        key = input("Enter key: ").lower()

        if key in threat:
            del threat[key]
            print(f"\n[Field deleted] Updated fields: {threat}")
    
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
    elif option == "5":
        key = input("Enter key: ").lower()

        if key in threat:
            print(f"\nField [{key}] exists.",
                  f"\nCurrent fields: {threat}",
                  sep="")
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")

### Revision 2: Bug Check - missing update flexibility and escape hatches
* Option 2 doesn't have an option to let users only update the `old_value`, forcing them to re-enter the `old_key` if it doesn't need changing.
* Option 2 and 3 forcefully flatten `new_value` inputs, which is a problem if the value has uppercase characters.
* Option 3 can accidentally let you enter an empty string if you hit `Enter` while typing for a new key.
* Option 2 and 3 don't have escape hatches in case a user changes their mind or made a typo while entering values.

**The fix:**
* Removed `.lower()` from option 2 and 3 when entering `new_value`.
* Restructured option 2 to give users the option to update the `old_key`.
* Added `.strip()` on all prompts to catch empty inputs.
* Allowed users to type `q` at prompts from Option 2 to 3 to cancel the current action and go back to the main menu using `continue`.

**Improvements:**
* Added `clear_screen` for cleaner output.
* Added `input("\nPress Enter to return to menu...")` to all option blocks so users can read output before the screen clears.

Updated script:


In [ ]:
import os

def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

threat = {
    "ip": "185.220.101.1",
    "country": "RU",
    "severity": "high",
    "alert_count": 5
}

while True:
    print("\n" + "="*41)
    print("="*9 + " THREAT RECORD MANAGER " + "="*9)
    print("="*41)
    print(" [1] View threat record\n",
          " [2] Update a field\n",
          " [3] Add a new field\n",
          " [4] Delete a field\n",
          " [5] Check if a field exists\n",
          " [Q] Quit\n",
          sep="")
    
    option = input("Enter option: ").lower()

    if option == "q":
        print("\nClosing...")
        break
        
# View fields
    elif option == "1":
        print(f"\n{threat}")
        
        input("\nPress Enter to return to menu...")
        clear_screen()

# Update a field
    elif option == "2":
        old_key = input("Enter key to update (or 'q' to cancel):").strip().lower()

        if old_key == 'q' or not old_key:
            print("\nAction canceled.")
            continue
            
        elif old_key in threat:
            print(f"Current value for '{old_key}': {threat[old_key]}")
            
            new_value = input("Enter new value (press Enter to keep current, or 'q' to cancel: ").strip()

            if new_value.lower() == 'q':
                print("\nAction canceled.")
                continue

            if new_value:
                threat[old_key] = new_value
                print(f"[Value updated]")
                
            rename_choice = input(f"Rename the old key '{old_key}'? (yes/no, or 'q' to cancel): ").strip().lower()

            if rename_choice == 'q':
                print("\nAction canceled.")
                continue

            if rename_choice == 'yes':
                new_key = input("\nEnter new key (or 'q' to cancel): ").strip().lower()
                
                if not new_key or new_key == 'q':
                    print("Renaming canceled.")
                    continue
                    
                elif new_key == old_key:
                    print("New key name matches the old one. No change made.")
                elif new_key in threat:
                    print(f"[ERROR]: The key '{new_key}' already exists. Rename aborted.")
                else:
                    threat[new_key] = threat[old_key]
                    del threat[old_key]
                    print(f"[Key renamed] '{old_key}' changed to '{new_key}' ")
                    
            print(f"\n[Field updated] Updated fields: {threat}")

        else:
            print(f"\nField {old_key} not found")

        input("\nPress Enter to return to menu...")
        clear_screen()

# Add a field
    elif option == "3":
        new_key = input("Enter new key (or 'q' to cancel): ").strip().lower()

        if new_key == 'q' or not new_key:
            print("\nAction canceled.")
            continue
            
        elif new_key not in threat:
            new_value = input("Enter new value (or 'q' to cancel): ").strip()

            if new_value == 'q' or not new_value:
                print("\nAction canceled.")
                continue

            else:
                threat[new_key] = new_value
                print(f"\n[Field added] Updated fields: {threat}")
            
        else:
            print("\nField already exists. Use option 2 to update.")

        input("\nPress Enter to return to menu...")
        clear_screen()

# Delete a field
    elif option == "4":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            del threat[key]
            print(f"\n[Field deleted] Updated fields: {threat}")
    
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
        input("\nPress Enter to return to menu...")
        clear_screen()

# Check a field
    elif option == "5":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            print(f"\nField [{key}] exists.",
                  f"\nCurrent fields: {threat}",
                  sep="")
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")

        input("\nPress Enter to return to menu...")
        clear_screen()

### Revision 3: Bug Check - unhandled input and premature screen clearing
* In Option 2, the rename prompt only handled `yes` and `q` but not `no`. If a user typed `no`, it would just print the final print statement.
* `clear_screen()` was placed at the bottom of each block. When I canceled an action with `continue`, it jumped back to the top and skipped `clear_screen()` entirely, which left the old text on the screen.

**The fix:**
* Added explicit `elif-else` blocks to handle `no` and any invalid input.
  * Every possible input is now explicitly handled. The user always gets clear feedback on what happened, and invalid inputs like `noo` or `n` are caught gracefully instead of silently doing nothing.
* Moved `clear_screen()` to the top of the `while True` loop so it runs automatically at the start of every iteration regardless of how the previous one ended.
  * This provides consistent screen clearing on every option chosen without duplicating `clear_screen()` inside every option block.

Updated script:


In [ ]:
import os

def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

threat = {
    "ip": "185.220.101.1",
    "country": "RU",
    "severity": "high",
    "alert_count": 5
}

while True:
    clear_screen()
    
    print("\n" + "="*41)
    print("="*9 + " THREAT RECORD MANAGER " + "="*9)
    print("="*41)
    print(" [1] View threat record\n",
          " [2] Update a field\n",
          " [3] Add a new field\n",
          " [4] Delete a field\n",
          " [5] Check if a field exists\n",
          " [Q] Quit\n",
          sep="")
    
    option = input("Enter option: ").lower()

    if option == "q":
        print("\nClosing...")
        break
        
# View fields
    elif option == "1":
        print(f"\n{threat}")
        
        input("\nPress Enter to return to menu...")
        
# Update a field
    elif option == "2":
        old_key = input("Enter key to update (or 'q' to cancel):").strip().lower()

        if old_key == 'q' or not old_key:
            print("\nAction canceled.")
            continue
            
        elif old_key in threat:
            print(f"Current value for '{old_key}': {threat[old_key]}")
            
            new_value = input("Enter new value (press Enter to keep current, or 'q' to cancel: ").strip()

            if new_value.lower() == 'q':
                print("\nAction canceled.")
                continue

            if new_value:
                threat[old_key] = new_value
                print(f"[Value updated]")
                
            rename_choice = input(f"Rename the old key '{old_key}'? (yes/no, or 'q' to cancel): ").strip().lower()

            if rename_choice == 'q':
                print("\nAction canceled.")
                continue

            elif rename_choice == 'yes':
                new_key = input("\nEnter new key (or 'q' to cancel): ").strip().lower()
                
                if not new_key or new_key == 'q':
                    print("Renaming canceled.")
                    continue
                    
                elif new_key == old_key:
                    print("New key name matches the old one. No change made.")
                elif new_key in threat:
                    print(f"[ERROR]: The key '{new_key}' already exists. Rename aborted.")
                
                else:
                    threat[new_key] = threat[old_key]
                    del threat[old_key]
                    print(f"[Key renamed] '{old_key}' changed to '{new_key}' ")
                    
            elif rename_choice == 'no':
                print("\nKey rename skipped")
                
            else:
                print("Invalid input. Key rename skipped.")
                
            print(f"\n[Field updated] Updated fields: {threat}")

        else:
            print(f"\nField {old_key} not found")

        input("\nPress Enter to return to menu...")

# Add a field
    elif option == "3":
        new_key = input("Enter new key (or 'q' to cancel): ").strip().lower()

        if new_key == 'q' or not new_key:
            print("\nAction canceled.")
            continue
            
        elif new_key not in threat:
            new_value = input("Enter new value (or 'q' to cancel): ").strip()

            if new_value == 'q' or not new_value:
                print("\nAction canceled.")
                continue

            else:
                threat[new_key] = new_value
                print(f"\n[Field added] Updated fields: {threat}")
            
        else:
            print("\nField already exists. Use option 2 to update.")

        input("\nPress Enter to return to menu...")

# Delete a field
    elif option == "4":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            del threat[key]
            print(f"\n[Field deleted] Updated fields: {threat}")
    
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
        input("\nPress Enter to return to menu...")

# Check a field
    elif option == "5":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            print(f"\nField [{key}] exists.",
                  f"\nCurrent fields: {threat}",
                  sep="")
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
        input("\nPress Enter to return to menu...")

### Revision 4: Bug Check - error messages cleared before they could be read
* (Option 2) When a rename was canceled or a key collision was detected, `continue` jumped straight to the top of the loop and `clear_screen()` wiped the error message before I could read it.
* I accidentally typed 6 at the main menu and it didn't return anything. No error message, no feedback, just a blank screen.

**The fix:**
* Added `input("\nPress Enter to return to menu...")` before each `continue` inside the rename block so the screen only clears after the user acknowledges the message.
  * Error and cancel messages now stay visible until the user is ready to continue, consistent with the UX pattern used throughout the rest of the application.
* Added an `else` block at the end of the script.
  * The script now responds to every possible input. Users who mistype or enter unexpected values get clear guidance instead of a confusing blank screen.

**Updates:**
* Changed `[Field updated] Updated fields:` to `[Current Field]:` for cleaner output labeling.

Final script:


In [ ]:
import os

def clear_screen():
    os.system('cls' if os.name == 'nt' else 'clear')

threat = {
    "ip": "185.220.101.1",
    "country": "RU",
    "severity": "high",
    "alert_count": 5
}

while True:
    clear_screen()
    
    print("\n" + "="*41)
    print("="*9 + " THREAT RECORD MANAGER " + "="*9)
    print("="*41)
    print(" [1] View threat record\n",
          " [2] Update a field\n",
          " [3] Add a new field\n",
          " [4] Delete a field\n",
          " [5] Check if a field exists\n",
          " [Q] Quit\n",
          sep="")
    
    option = input("Enter option: ").lower()

    if option == "q":
        print("\nClosing...")
        break
        
# View fields
    elif option == "1":
        print(f"\n{threat}")
        
        input("\nPress Enter to return to menu...")
        
# Update a field
    elif option == "2":
        old_key = input("Enter key to update (or 'q' to cancel):").strip().lower()

        if old_key == 'q' or not old_key:
            print("\nAction canceled.")
            continue
            
        elif old_key in threat:
            print(f"Current value for '{old_key}': {threat[old_key]}")
            
            new_value = input("Enter new value (press Enter to keep current, or 'q' to cancel: ").strip()

            if new_value.lower() == 'q':
                print("\nAction canceled.")
                continue

            if new_value:
                threat[old_key] = new_value
                print(f"[Value updated]")
                
            rename_choice = input(f"Rename the old key '{old_key}'? (yes/no, or 'q' to cancel): ").strip().lower()

            if rename_choice == 'q':
                print("\nAction canceled.")
                continue

            elif rename_choice == 'yes':
                new_key = input("\nEnter new key (or 'q' to cancel): ").strip().lower()
                
                if not new_key or new_key == 'q':
                    print("Renaming canceled.")
                    input("\nPress Enter to return to menu...")
                    continue
                    
                elif new_key == old_key:
                    print("New key name matches the old one. No change made.")
                elif new_key in threat:
                    print(f"[ERROR]: The key '{new_key}' already exists. Rename aborted.")
                    input("\nPress Enter to return to menu...")
                    continue
                    
                else:
                    threat[new_key] = threat[old_key]
                    del threat[old_key]
                    print(f"[Key renamed] '{old_key}' changed to '{new_key}' ")
                    
            elif rename_choice == 'no':
                print("\nKey rename skipped")
                
            else:
                print("Invalid input. Key rename skipped.")
                
            print(f"\n[Current Field]: {threat}")

        else:
            print(f"\nField {old_key} not found")

        input("\nPress Enter to return to menu...")

# Add a field
    elif option == "3":
        new_key = input("Enter new key (or 'q' to cancel): ").strip().lower()

        if new_key == 'q' or not new_key:
            print("\nAction canceled.")
            continue
            
        elif new_key not in threat:
            new_value = input("Enter new value (or 'q' to cancel): ").strip()

            if new_value == 'q' or not new_value:
                print("\nAction canceled.")
                continue

            else:
                threat[new_key] = new_value
                print(f"\n[Field added] Updated fields: {threat}")
            
        else:
            print("\nField already exists. Use option 2 to update.")

        input("\nPress Enter to return to menu...")

# Delete a field
    elif option == "4":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            del threat[key]
            print(f"\n[Field deleted] Updated fields: {threat}")
    
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
        input("\nPress Enter to return to menu...")

# Check a field
    elif option == "5":
        key = input("Enter key: ").strip().lower()

        if key in threat:
            print(f"\nField [{key}] exists.",
                  f"\nCurrent fields: {threat}",
                  sep="")
        else:
            print("\nField doesn't exist. Use option 3 to add or try again.")
            
        input("\nPress Enter to return to menu...")
        
    else:
        print("\nInvalid option. Please choose 1-5 or Q.")
        input("\nPress Enter to return to menu...")

## Closing Notes

There's no live dataset to report findings on here since this tool manages a single hardcoded record, but the build itself demonstrates the core defensive-programming habits a SOC analyst's internal tools need: every menu path handles bad input explicitly instead of relying on `try/except` as a catch-all, every action can be canceled mid-flow without leaving the screen in an inconsistent state, and destructive operations (rename, delete) confirm before acting.

The four revisions above trace a consistent pattern: each bug was caught by actually running the tool and trying to break it, not by reading the code and guessing. That habit, run it, try to break it, read the error, fix the root cause, is the one worth carrying into every project after this one.
